# CSE422 — Telco Customer Churn Prediction (Merged Notebook)

In [ ]:
from pathlib import Path
DATA_PATH = 'telco_customer_churn.csv'
MODELS_DIR = Path('models')
MODELS_DIR.mkdir(exist_ok=True)
if not Path(DATA_PATH).exists():
    try:
        from google.colab import files
        print(f"'{DATA_PATH}' not found in this Colab session -- please upload it now:")
        uploaded = files.upload()
    except ImportError:
        raise FileNotFoundError(f"'{DATA_PATH}' not found. Please upload it to the working directory.")

In [ ]:
from __future__ import annotations
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
RANDOM_STATE = 42
TARGET_COL = 'Churn'
ID_COL = 'customerID'
NUMERIC_FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges']
BINARY_FEATURES = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
MULTI_CATEGORY_FEATURES = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

def load_raw_data(path: str) -> pd.DataFrame:
    return pd.read_csv(path)

def count_blank_total_charges(df: pd.DataFrame) -> int:
    return int(pd.to_numeric(df['TotalCharges'], errors='coerce').isna().sum())

def fix_total_charges(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(0.0)
    return df

def normalize_senior_citizen(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
    return df

def drop_identifier(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=[ID_COL])

def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df = fix_total_charges(df)
    df = normalize_senior_citizen(df)
    df = drop_identifier(df)
    return df

def split_features_target(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    X = df.drop(columns=[TARGET_COL])
    y = df[TARGET_COL].map({'No': 0, 'Yes': 1})
    return (X, y)

def train_test_split_data(X: pd.DataFrame, y: pd.Series, test_size: float=0.2, random_state: int=RANDOM_STATE):
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)

def build_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(transformers=[('num', StandardScaler(), NUMERIC_FEATURES), ('bin', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), BINARY_FEATURES), ('nom', OneHotEncoder(handle_unknown='ignore'), MULTI_CATEGORY_FEATURES)], remainder='drop')

def get_feature_names_out(preprocessor: ColumnTransformer) -> list[str]:
    return list(preprocessor.get_feature_names_out())

In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Any
import joblib
import pandas as pd
DEFAULT_MODELS_DIR = Path('models')
MODEL_FILES = {'Decision Tree': 'decision_tree_model.joblib', 'Logistic Regression': 'logistic_regression_model.joblib', 'Neural Network': 'neural_network_model.joblib'}
MODEL_DISPLAY_ORDER = ['Decision Tree', 'Logistic Regression', 'Neural Network']
RAW_INPUT_COLUMNS = ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']

def load_artifacts(models_dir: Path | str=DEFAULT_MODELS_DIR) -> dict[str, Any]:
    models_dir = Path(models_dir)
    preprocessor_path = models_dir / 'preprocessor.joblib'
    if not preprocessor_path.exists():
        raise FileNotFoundError(f'{preprocessor_path} not found. Run `src/train_and_save_models.py` first to generate the model artifacts.')
    preprocessor = joblib.load(preprocessor_path)
    models: dict[str, Any] = {}
    for name, filename in MODEL_FILES.items():
        model_path = models_dir / filename
        if not model_path.exists():
            raise FileNotFoundError(f'{model_path} not found. Run `src/train_and_save_models.py` first to generate the model artifacts.')
        models[name] = joblib.load(model_path)
    return {'preprocessor': preprocessor, 'models': models}

def _ensure_customer_id(record: dict) -> dict:
    record = dict(record)
    record.setdefault(ID_COL, 'NEW-CUSTOMER')
    return record

def preprocess_new_customer(record: dict, preprocessor: Any):
    record = _ensure_customer_id(record)
    missing = [c for c in RAW_INPUT_COLUMNS if c not in record]
    if missing:
        raise ValueError(f'Missing required field(s) for a new customer record: {missing}')
    df = pd.DataFrame([record], columns=RAW_INPUT_COLUMNS)
    df = fix_total_charges(df)
    df = normalize_senior_citizen(df)
    df = drop_identifier(df)
    return preprocessor.transform(df)

def predict_all_models(record: dict, artifacts: dict) -> dict[str, dict[str, Any]]:
    X = preprocess_new_customer(record, artifacts['preprocessor'])
    results: dict[str, dict[str, Any]] = {}
    for name in MODEL_DISPLAY_ORDER:
        model = artifacts['models'][name]
        pred = model.predict(X)[0]
        proba = model.predict_proba(X)[0, 1]
        results[name] = {'prediction': 'Yes' if pred == 1 else 'No', 'probability_churn': float(proba)}
    return results

def format_prediction_report(record: dict, results: dict[str, dict[str, Any]]) -> str:
    customer_id = record.get(ID_COL, 'NEW-CUSTOMER')
    lines = [f'Churn prediction for customer: {customer_id}', '-' * 50]
    for name in MODEL_DISPLAY_ORDER:
        r = results[name]
        lines.append(f'  {name:22s} -> Prediction: {r['prediction']:3s}   P(Churn) = {r['probability_churn']:.4f}')
    return '\n'.join(lines)

# Stage 1 — Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.facecolor'] = 'white'
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [ ]:
DATA_PATH = 'telco_customer_churn.csv'
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
n_rows, n_cols = df.shape
print(f'Number of data points (rows): {n_rows}')
print(f'Number of columns (total, incl. ID and target): {n_cols}')
print(f'Number of input features (excl. customerID and target Churn): {n_cols - 2}')

In [ ]:
print('Unique values of Churn:', df['Churn'].unique())
print('Dtype of Churn:', df['Churn'].dtype)
print()
print(df['Churn'].value_counts())

In [ ]:
feature_info = pd.DataFrame({'dtype': df.dtypes.astype(str), 'n_unique': df.nunique(), 'sample_values': [df[c].unique()[:4].tolist() for c in df.columns]})
feature_info

In [ ]:
from sklearn.preprocessing import LabelEncoder
df_corr = df.drop(columns=['customerID']).copy()
df_corr['TotalCharges'] = pd.to_numeric(df_corr['TotalCharges'], errors='coerce')
n_dropped = int(df_corr['TotalCharges'].isna().sum())
print(f'Rows excluded from this correlation computation only (blank TotalCharges): {n_dropped}')
df_corr = df_corr.dropna(subset=['TotalCharges'])
categorical_cols = df_corr.select_dtypes(include='object').columns.tolist()
print(f'Columns label-encoded for this heatmap only: {categorical_cols}')
le = LabelEncoder()
for col in categorical_cols:
    df_corr[col] = le.fit_transform(df_corr[col])
corr_matrix = df_corr.corr()

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, linewidths=0.3, cbar_kws={'label': 'Pearson correlation'})
plt.title('Correlation Heatmap — All Features (label-encoded) and Target (Churn)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
churn_corr = corr_matrix['Churn'].drop('Churn').sort_values(key=abs, ascending=False)
print('Features ranked by |correlation| with Churn:')
churn_corr

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100
print(churn_counts)
print()
print(churn_pct.round(2).astype(str) + '%')

In [ ]:
plt.figure(figsize=(6, 5))
colors = ['#4C72B0', '#DD8452']
ax = sns.barplot(x=churn_counts.index, y=churn_counts.values, palette=colors, hue=churn_counts.index, legend=False)
ax.set_title('Churn Class Distribution (N = 2 classes)', fontsize=13)
ax.set_xlabel('Churn')
ax.set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    ax.text(i, v + 40, f'{v} ({churn_pct.values[i]:.1f}%)', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
contract_churn = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
order = contract_churn['Yes'].sort_values(ascending=False).index
plt.figure(figsize=(7, 5))
sns.barplot(x=order, y=contract_churn.loc[order, 'Yes'], color='#C44E52')
plt.title('Churn Rate (%) by Contract Type')
plt.xlabel('Contract Type')
plt.ylabel('Churn Rate (%)')
plt.tight_layout()
plt.show()
contract_churn.round(2)

In [ ]:
internet_churn = pd.crosstab(df['InternetService'], df['Churn'], normalize='index') * 100
order = internet_churn['Yes'].sort_values(ascending=False).index
plt.figure(figsize=(7, 5))
sns.barplot(x=order, y=internet_churn.loc[order, 'Yes'], color='#55A868')
plt.title('Churn Rate (%) by Internet Service Type')
plt.xlabel('Internet Service')
plt.ylabel('Churn Rate (%)')
plt.tight_layout()
plt.show()
internet_churn.round(2)

In [ ]:
payment_churn = pd.crosstab(df['PaymentMethod'], df['Churn'], normalize='index') * 100
order = payment_churn['Yes'].sort_values(ascending=False).index
plt.figure(figsize=(8, 5))
sns.barplot(x=order, y=payment_churn.loc[order, 'Yes'], color='#8172B2')
plt.title('Churn Rate (%) by Payment Method')
plt.xlabel('Payment Method')
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()
payment_churn.round(2)

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='tenure', hue='Churn', bins=30, kde=True, element='step')
plt.title('Tenure Distribution by Churn Status')
plt.xlabel('Tenure (months)')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', hue='Churn', legend=False)
plt.title('Monthly Charges by Churn Status')
plt.xlabel('Churn')
plt.ylabel('Monthly Charges ($)')
plt.tight_layout()
plt.show()

In [ ]:
senior_churn = pd.crosstab(df['SeniorCitizen'], df['Churn'], normalize='index') * 100
senior_churn.index = senior_churn.index.map({0: 'Not Senior', 1: 'Senior Citizen'})
senior_churn.round(2)

# Stage 2 — Preprocessing

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [ ]:
DATA_PATH = 'telco_customer_churn.csv'
df_raw = load_raw_data(DATA_PATH)
print('Raw shape:', df_raw.shape)
df_raw.head(3)

In [ ]:
print('Raw dtype of TotalCharges:', df_raw['TotalCharges'].dtype)
print('NaNs reported by .isna() (misleading):', df_raw['TotalCharges'].isna().sum())
n_blank = count_blank_total_charges(df_raw)
print(f'Rows that fail numeric conversion (true missing values): {n_blank}')
blank_mask = pd.to_numeric(df_raw['TotalCharges'], errors='coerce').isna()
df_raw.loc[blank_mask, ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']]

In [ ]:
df_step1 = fix_total_charges(df_raw)
print('Dtype after fix:', df_step1['TotalCharges'].dtype)
print('Remaining missing values:', df_step1['TotalCharges'].isna().sum())
df_step1.loc[blank_mask, ['customerID', 'tenure', 'TotalCharges']]

In [ ]:
print('SeniorCitizen unique values (before):', sorted(df_step1['SeniorCitizen'].unique()))
print('SeniorCitizen dtype (before):', df_step1['SeniorCitizen'].dtype)

In [ ]:
df_step2 = normalize_senior_citizen(df_step1)
print('SeniorCitizen unique values (after):', sorted(df_step2['SeniorCitizen'].unique()))
print('SeniorCitizen dtype (after):', df_step2['SeniorCitizen'].dtype)

In [ ]:
print('Total rows:', len(df_step2))
print('Unique customerID values:', df_step2['customerID'].nunique())

In [ ]:
df_clean = drop_identifier(df_step2)
print('Columns after dropping customerID:', df_clean.shape[1])
df_clean.head(3)

In [ ]:
assert clean_dataset(df_raw).equals(df_clean)
print('clean_dataset() reproducibility check passed.')

In [ ]:
categorical_cols = df_clean.drop(columns=[TARGET_COL]).select_dtypes(include='object').columns.tolist()
print(f'Categorical input columns requiring encoding ({len(categorical_cols)}):')
print(categorical_cols)
print()
print('Binary (2-level) features:', BINARY_FEATURES)
print()
print('Multi-level (3+) nominal features:', MULTI_CATEGORY_FEATURES)

In [ ]:
df_clean[NUMERIC_FEATURES].describe().loc[['min', 'max', 'mean', 'std']]

In [ ]:
X, y = split_features_target(df_clean)
X_train, X_test, y_train, y_test = train_test_split_data(X, y)
print('X_train:', X_train.shape, ' X_test:', X_test.shape)
print()
print('y_train class balance:')
print(y_train.value_counts(normalize=True).rename({0: 'No (0)', 1: 'Yes (1)'}).round(4))
print()
print('y_test class balance:')
print(y_test.value_counts(normalize=True).rename({0: 'No (0)', 1: 'Yes (1)'}).round(4))

In [ ]:
preprocessor = build_preprocessor()
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
feature_names = get_feature_names_out(preprocessor)
print('X_train_processed shape:', X_train_processed.shape)
print('X_test_processed shape: ', X_test_processed.shape)
print('Number of output features after encoding:', len(feature_names))

In [ ]:
X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_train_processed_df.head(3)

In [ ]:
print('Missing values in X_train_processed:', np.isnan(X_train_processed).sum())
print('Missing values in X_test_processed: ', np.isnan(X_test_processed).sum())

In [ ]:
num_out_cols = [c for c in feature_names if c.startswith('num__')]
train_num_df = X_train_processed_df[num_out_cols]
print('Train scaled numeric features -- mean (should be ~0):')
print(train_num_df.mean().round(3))
print()
print('Train scaled numeric features -- std (should be ~1):')
print(train_num_df.std().round(3))

In [ ]:
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)
test_num_df = X_test_processed_df[num_out_cols]
print('Test scaled numeric features -- mean (expected close to, but not exactly, 0):')
print(test_num_df.mean().round(3))
print()
print('Test scaled numeric features -- std (expected close to, but not exactly, 1):')
print(test_num_df.std().round(3))

# Stage 3 — Supervised Models

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [ ]:
DATA_PATH = 'telco_customer_churn.csv'
df_raw = load_raw_data(DATA_PATH)
df_clean = clean_dataset(df_raw)
X, y = split_features_target(df_clean)
X_train, X_test, y_train, y_test = train_test_split_data(X, y)
preprocessor = build_preprocessor()
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
feature_names = get_feature_names_out(preprocessor)
print('X_train_processed:', X_train_processed.shape)
print('X_test_processed: ', X_test_processed.shape)
print()
print('y_train class balance:')
print(y_train.value_counts(normalize=True).rename({0: 'No (0)', 1: 'Yes (1)'}).round(4))
print()
print('y_test class balance:')
print(y_test.value_counts(normalize=True).rename({0: 'No (0)', 1: 'Yes (1)'}).round(4))

In [ ]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_score)
    cm = confusion_matrix(y_test, y_pred)
    print(f'=== {name} ===')
    print(f'Accuracy : {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall   : {rec:.4f}')
    print(f'AUC      : {auc:.4f}')
    print()
    print('Confusion Matrix (rows = actual, cols = predicted, order = [No, Yes]):')
    print(cm)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No', 'Yes']).plot(ax=axes[0], cmap='Blues', colorbar=False)
    axes[0].set_title(f'{name} — Confusion Matrix')
    fpr, tpr, _ = roc_curve(y_test, y_score)
    axes[1].plot(fpr, tpr, color='#4C72B0', label=f'AUC = {auc:.3f}')
    axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'{name} — ROC Curve')
    axes[1].legend(loc='lower right')
    plt.tight_layout()
    plt.show()
    return {'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'AUC': auc, 'y_pred': y_pred, 'y_score': y_score, 'confusion_matrix': cm}
results = {}

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
results['Decision Tree'] = evaluate_model('Decision Tree', dt_model, X_train_processed, y_train, X_test_processed, y_test)

In [ ]:
logreg_model = LogisticRegression(max_iter=1000, random_state=42)
results['Logistic Regression'] = evaluate_model('Logistic Regression', logreg_model, X_train_processed, y_train, X_test_processed, y_test)

In [ ]:
mlp_model = MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu', max_iter=500, early_stopping=True, random_state=42)
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter('always')
    results['Neural Network'] = evaluate_model('Neural Network', mlp_model, X_train_processed, y_train, X_test_processed, y_test)
if caught_warnings:
    print(f'\n{len(caught_warnings)} warning(s) raised while training the Neural Network:')
    for w in caught_warnings:
        print(f'  - {w.category.__name__}: {w.message}')
else:
    print('\nNo warnings raised while training the Neural Network.')

In [ ]:
summary_df = pd.DataFrame([{'Model': r['Model'], 'Accuracy': r['Accuracy'], 'Precision': r['Precision'], 'Recall': r['Recall'], 'AUC': r['AUC']} for r in results.values()]).set_index('Model').round(4)
summary_df

In [ ]:
assert summary_df.notna().all().all(), 'Missing metric(s) detected!'
assert list(summary_df.index) == ['Decision Tree', 'Logistic Regression', 'Neural Network']
for r in results.values():
    assert r['confusion_matrix'].shape == (2, 2)
    assert len(r['y_score']) == len(y_test)
print('All 3 models trained and evaluated successfully.')
print('Accuracy, Precision, Recall, Confusion Matrix, and AUC/ROC were computed for every model.')

# Stage 4 — K-Means Clustering

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
RANDOM_STATE = 42

In [ ]:
DATA_PATH = 'telco_customer_churn.csv'
df_raw = load_raw_data(DATA_PATH)
df_clean = clean_dataset(df_raw)
X, y_churn = split_features_target(df_clean)
print('X shape (features only, Churn excluded):', X.shape)
print('Columns in X:', list(X.columns))
assert 'Churn' not in X.columns, 'Churn must not be present in the clustering feature matrix!'

In [ ]:
preprocessor = build_preprocessor()
X_processed = preprocessor.fit_transform(X)
feature_names = get_feature_names_out(preprocessor)
print('X_processed shape:', X_processed.shape)
print('Number of processed features:', len(feature_names))

In [ ]:
k_range = list(range(2, 11))
inertias = []
silhouette_scores = []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels_k = km.fit_predict(X_processed)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_processed, labels_k)
    silhouette_scores.append(sil)
    print(f'K={k:2d}  |  inertia={km.inertia_:10.2f}  |  silhouette={sil:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(k_range, inertias, marker='o', color='#4C72B0')
axes[0].set_title('Elbow Method — Inertia vs. K')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (within-cluster sum of squares)')
axes[0].set_xticks(k_range)
axes[1].plot(k_range, silhouette_scores, marker='o', color='#C44E52')
axes[1].set_title('Silhouette Score vs. K')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Average Silhouette Score')
axes[1].set_xticks(k_range)
plt.tight_layout()
plt.show()

In [ ]:
best_k_silhouette = k_range[int(np.argmax(silhouette_scores))]
best_silhouette_value = max(silhouette_scores)
pct_reductions = [(inertias[i - 1] - inertias[i]) / inertias[i - 1] for i in range(1, len(inertias))]
elbow_candidates = [k_range[i + 1] for i, pct in enumerate(pct_reductions) if pct < 0.1]
elbow_k = elbow_candidates[0] if elbow_candidates else k_range[-1]
print(f'K that maximizes silhouette score : {best_k_silhouette}  (silhouette = {best_silhouette_value:.4f})')
print(f'Elbow heuristic suggests K        : {elbow_k}  (first K with <10% further inertia reduction)')
print()
print('Per-K marginal inertia reduction:')
for k, pct in zip(k_range[1:], pct_reductions):
    print(f'  K={k:2d}: {pct * 100:5.1f}% reduction from previous K')

In [ ]:
SELECTED_K = best_k_silhouette
agreement = 'agrees with' if SELECTED_K == elbow_k else 'differs from'
print(f'SELECTED K = {SELECTED_K}')
print(f"(Silhouette-maximizing K {agreement} the elbow heuristic's suggestion of K={elbow_k}.)")
print()
print('Decision rule applied: choose the silhouette-maximizing K as the primary, objective criterion; the elbow curve is used only as a qualitative sanity check, since inertia alone cannot identify an optimal K on its own.')

In [ ]:
kmeans_final = KMeans(n_clusters=SELECTED_K, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_processed)
cluster_sizes = pd.Series(cluster_labels).value_counts().sort_index()
cluster_pct = (cluster_sizes / len(cluster_labels) * 100).round(2)
cluster_size_table = pd.DataFrame({'Count': cluster_sizes, 'Percent': cluster_pct})
cluster_size_table.index.name = 'Cluster'
cluster_size_table

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_processed)
explained = pca.explained_variance_ratio_
print(f'Explained variance -- PC1: {explained[0] * 100:.1f}%, PC2: {explained[1] * 100:.1f}%, total: {explained.sum() * 100:.1f}%')

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='tab10', s=12, alpha=0.7)
plt.title(f'K-Means Clusters (K={SELECTED_K}) — Visualized via PCA (2 Components)')
plt.xlabel(f'Principal Component 1 ({explained[0] * 100:.1f}% variance)')
plt.ylabel(f'Principal Component 2 ({explained[1] * 100:.1f}% variance)')
legend1 = plt.legend(*scatter.legend_elements(), title='Cluster', loc='best')
plt.gca().add_artist(legend1)
plt.tight_layout()
plt.show()

In [ ]:
df_clusters = df_clean.copy()
df_clusters['Cluster'] = cluster_labels
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
sc0 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='tab10', s=12, alpha=0.7)
axes[0].set_title(f'K-Means Clusters (K={SELECTED_K})')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
leg0 = axes[0].legend(*sc0.legend_elements(), title='Cluster', loc='best')
axes[0].add_artist(leg0)
churn_colors = df_clusters['Churn'].map({'No': 0, 'Yes': 1})
sc1 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=churn_colors, cmap='coolwarm', s=12, alpha=0.7)
axes[1].set_title('Actual Churn Labels (post-hoc reference only)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=plt.cm.coolwarm(0.0), markersize=8, label='No'), plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=plt.cm.coolwarm(1.0), markersize=8, label='Yes')]
axes[1].legend(handles=handles, title='Churn', loc='best')
plt.tight_layout()
plt.show()

In [ ]:
churn_by_cluster = pd.crosstab(df_clusters['Cluster'], df_clusters['Churn'], normalize='index') * 100
churn_by_cluster = churn_by_cluster.round(2)
churn_by_cluster['Cluster Size'] = cluster_sizes.values
churn_by_cluster

In [ ]:
numeric_summary = df_clusters.groupby('Cluster')[['tenure', 'MonthlyCharges', 'TotalCharges']].mean().round(2)
numeric_summary

In [ ]:
def top_category_pct(df, cluster_col, feature_col):
    rows = []
    for c, group in df.groupby(cluster_col):
        vc = group[feature_col].value_counts(normalize=True)
        rows.append({'Cluster': c, 'Most common': vc.index[0], 'Share (%)': round(vc.iloc[0] * 100, 1)})
    return pd.DataFrame(rows).set_index('Cluster')
for col in ['Contract', 'InternetService', 'PaymentMethod']:
    print(f'--- Most common {col} per cluster ---')
    print(top_category_pct(df_clusters, 'Cluster', col))
    print()

In [ ]:
senior_by_cluster = df_clusters.groupby('Cluster')['SeniorCitizen'].apply(lambda s: (s == 'Yes').mean() * 100).round(2).rename('Senior Citizen (%)')
senior_by_cluster

# Stage 5 — Model Comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [ ]:
DATA_PATH = 'telco_customer_churn.csv'
df_raw = load_raw_data(DATA_PATH)
df_clean = clean_dataset(df_raw)
X, y = split_features_target(df_clean)
X_train, X_test, y_train, y_test = train_test_split_data(X, y)
preprocessor = build_preprocessor()
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
print('X_train_processed:', X_train_processed.shape, ' X_test_processed:', X_test_processed.shape)

In [ ]:
MODEL_CONFIGS = {'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42), 'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42), 'Neural Network': MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu', max_iter=500, early_stopping=True, random_state=42)}
results = {}
for name, model in MODEL_CONFIGS.items():
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    y_score = model.predict_proba(X_test_processed)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_score)
    results[name] = {'Accuracy': accuracy_score(y_test, y_pred), 'Precision': precision_score(y_test, y_pred), 'Recall': recall_score(y_test, y_pred), 'AUC': roc_auc_score(y_test, y_score), 'confusion_matrix': confusion_matrix(y_test, y_pred), 'fpr': fpr, 'tpr': tpr, 'y_pred': y_pred, 'y_score': y_score}
print('All 3 models trained and evaluated on the identical test set.')

In [ ]:
REFERENCE_RESULTS = {'Decision Tree': {'Accuracy': 0.7984, 'Precision': 0.6347, 'Recall': 0.5668, 'AUC': 0.8297}, 'Logistic Regression': {'Accuracy': 0.8055, 'Precision': 0.6572, 'Recall': 0.5588, 'AUC': 0.842}, 'Neural Network': {'Accuracy': 0.7999, 'Precision': 0.6361, 'Recall': 0.5749, 'AUC': 0.8389}}
verification_rows = []
all_match = True
for name, ref in REFERENCE_RESULTS.items():
    for metric, ref_value in ref.items():
        computed_value = round(results[name][metric], 4)
        diff = abs(computed_value - ref_value)
        match = diff <= 0.0005
        all_match &= match
        verification_rows.append({'Model': name, 'Metric': metric, '03_supervised_models.ipynb': ref_value, 'This notebook': computed_value, 'Match': 'Yes' if match else 'NO -- MISMATCH'})
verification_df = pd.DataFrame(verification_rows)
print(verification_df.to_string(index=False))
print()
print('ALL METRICS MATCH 03_supervised_models.ipynb:', all_match)
assert all_match, 'Recomputed metrics diverge from 03_supervised_models.ipynb -- investigate before proceeding!'

In [ ]:
comparison_df = pd.DataFrame([{'Model': name, 'Accuracy': r['Accuracy'], 'Precision': r['Precision'], 'Recall': r['Recall'], 'AUC': r['AUC']} for name, r in results.items()]).set_index('Model').round(4)
comparison_df

In [ ]:
plt.figure(figsize=(7, 4.5))
order = comparison_df['Accuracy'].sort_values(ascending=False)
sns.barplot(x=order.values, y=order.index, hue=order.index, palette='Blues_r', legend=False)
plt.title('Prediction Accuracy by Model')
plt.xlabel('Accuracy')
plt.ylabel('')
plt.xlim(0, 1)
for i, v in enumerate(order.values):
    plt.text(v + 0.01, i, f'{v:.4f}', va='center')
plt.tight_layout()
plt.show()

In [ ]:
pr_df = comparison_df[['Precision', 'Recall']].reset_index().melt(id_vars='Model', var_name='Metric', value_name='Score')
plt.figure(figsize=(7, 4.5))
sns.barplot(data=pr_df, x='Model', y='Score', hue='Metric', palette=['#4C72B0', '#DD8452'])
plt.title('Precision vs. Recall by Model')
plt.ylabel('Score')
plt.xlabel('')
plt.xticks(rotation=10, ha='right')
plt.ylim(0, 1)
plt.legend(title='')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, r) in zip(axes, results.items()):
    ConfusionMatrixDisplay(confusion_matrix=r['confusion_matrix'], display_labels=['No', 'Yes']).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(name)
plt.suptitle('Confusion Matrices — All 3 Models (rows = actual, cols = predicted)', y=1.04, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7.5, 6.5))
colors = sns.color_palette('tab10', n_colors=len(results))
for (name, r), color in zip(results.items(), colors):
    plt.plot(r['fpr'], r['tpr'], color=color, label=f'{name} (AUC = {r['AUC']:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess (AUC = 0.500)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All 3 Models')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4.5))
order_auc = comparison_df['AUC'].sort_values(ascending=False)
sns.barplot(x=order_auc.values, y=order_auc.index, hue=order_auc.index, palette='Greens_r', legend=False)
plt.title('AUC Score by Model')
plt.xlabel('AUC')
plt.ylabel('')
plt.xlim(0, 1)
for i, v in enumerate(order_auc.values):
    plt.text(v + 0.01, i, f'{v:.4f}', va='center')
plt.tight_layout()
plt.show()

In [ ]:
best_per_metric = comparison_df.idxmax()
best_values = comparison_df.max()
best_summary = pd.DataFrame({'Best Model': best_per_metric, 'Score': best_values.round(4)})
best_summary

# Train & Save Model Artifacts

In [ ]:
from __future__ import annotations
from pathlib import Path
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
DATA_PATH = Path('telco_customer_churn.csv')
MODELS_DIR = Path('models')
REFERENCE_RESULTS = {'Decision Tree': {'Accuracy': 0.7984, 'Precision': 0.6347, 'Recall': 0.5668, 'AUC': 0.8297}, 'Logistic Regression': {'Accuracy': 0.8055, 'Precision': 0.6572, 'Recall': 0.5588, 'AUC': 0.842}, 'Neural Network': {'Accuracy': 0.7999, 'Precision': 0.6361, 'Recall': 0.5749, 'AUC': 0.8389}}
TOLERANCE = 0.0005

def main() -> int:
    print('=' * 78)
    print('Reproducing the exact Stage-5 pipeline (src/preprocessing.py, unchanged)')
    print('=' * 78)
    df_raw = load_raw_data(str(DATA_PATH))
    df_clean = clean_dataset(df_raw)
    X, y = split_features_target(df_clean)
    X_train, X_test, y_train, y_test = train_test_split_data(X, y)
    preprocessor = build_preprocessor()
    X_train_processed = preprocessor.fit_transform(X_train)
    X_test_processed = preprocessor.transform(X_test)
    print(f'X_train_processed: {X_train_processed.shape}   X_test_processed: {X_test_processed.shape}')
    print()
    model_configs = {'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42), 'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42), 'Neural Network': MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu', max_iter=500, early_stopping=True, random_state=42)}
    fitted_models = {}
    computed_results = {}
    for name, model in model_configs.items():
        model.fit(X_train_processed, y_train)
        y_pred = model.predict(X_test_processed)
        y_score = model.predict_proba(X_test_processed)[:, 1]
        computed_results[name] = {'Accuracy': accuracy_score(y_test, y_pred), 'Precision': precision_score(y_test, y_pred), 'Recall': recall_score(y_test, y_pred), 'AUC': roc_auc_score(y_test, y_score)}
        fitted_models[name] = model
    print('=' * 78)
    print('Verifying reproduced metrics against the established Stage-5 reference values')
    print('=' * 78)
    all_match = True
    for name, ref in REFERENCE_RESULTS.items():
        for metric, ref_value in ref.items():
            computed_value = round(computed_results[name][metric], 4)
            diff = abs(computed_value - ref_value)
            match = diff <= TOLERANCE
            all_match &= match
            status = 'OK      ' if match else 'MISMATCH'
            print(f'  [{status}] {name:22s} {metric:10s} reference={ref_value:.4f}  computed={computed_value:.4f}')
    print()
    if not all_match:
        print('ABORTING: one or more metrics did not match the established Stage-5 reference')
        print('values. No artifacts were saved. Investigate the discrepancy before retrying.')
        return 1
    print('All metrics matched exactly. Saving artifacts...')
    MODELS_DIR.mkdir(exist_ok=True)
    joblib.dump(preprocessor, MODELS_DIR / 'preprocessor.joblib')
    joblib.dump(fitted_models['Decision Tree'], MODELS_DIR / 'decision_tree_model.joblib')
    joblib.dump(fitted_models['Logistic Regression'], MODELS_DIR / 'logistic_regression_model.joblib')
    joblib.dump(fitted_models['Neural Network'], MODELS_DIR / 'neural_network_model.joblib')
    for fname in ['preprocessor.joblib', 'decision_tree_model.joblib', 'logistic_regression_model.joblib', 'neural_network_model.joblib']:
        fpath = MODELS_DIR / fname
        print(f'  Saved: {fpath}  ({fpath.stat().st_size:,} bytes)')
    print()
    print('Done. src/predict.py can now load these artifacts.')
    return 0
_exit_code = main()
assert _exit_code == 0, 'train_and_save_models step failed -- see output above'

# Stage 6 — Predicting Churn for a New Customer

In [ ]:
import pandas as pd

In [ ]:
artifacts = load_artifacts()
print('Loaded preprocessor:', type(artifacts['preprocessor']).__name__)
print('Loaded models:')
for name in MODEL_DISPLAY_ORDER:
    print(f'  - {name}: {type(artifacts['models'][name]).__name__}')

In [ ]:
df_raw = pd.read_csv('telco_customer_churn.csv')
churner_row = df_raw[df_raw['customerID'] == '3668-QPYBK'].iloc[0]
nonchurner_row = df_raw[df_raw['customerID'] == '7590-VHVEG'].iloc[0]

def row_to_record(row):
    record = row.drop(labels=['Churn'], errors='ignore').to_dict()
    return record
print('=' * 60)
print(f'Real customer 3668-QPYBK -- actual historical Churn = {churner_row['Churn']} (reference only)')
print('=' * 60)
record = row_to_record(churner_row)
results = predict_all_models(record, artifacts)
print(format_prediction_report(record, results))

In [ ]:
print('=' * 60)
print(f'Real customer 7590-VHVEG -- actual historical Churn = {nonchurner_row['Churn']} (reference only)')
print('=' * 60)
record = row_to_record(nonchurner_row)
results = predict_all_models(record, artifacts)
print(format_prediction_report(record, results))

In [ ]:
edge_case_row = df_raw[df_raw['customerID'] == '4472-LVYGI'].iloc[0]
print('Raw TotalCharges value for this customer:', repr(edge_case_row['TotalCharges']))
print('tenure:', edge_case_row['tenure'])
print()
record = row_to_record(edge_case_row)
results = predict_all_models(record, artifacts)
print(format_prediction_report(record, results))
print()
print('No errors -- the blank-TotalCharges / tenure=0 case is handled correctly.')

In [ ]:
synthetic_new_customer = {'customerID': 'SYN-NEW-0001', 'gender': 'Male', 'SeniorCitizen': 0, 'Partner': 'No', 'Dependents': 'No', 'tenure': 0, 'PhoneService': 'Yes', 'MultipleLines': 'No', 'InternetService': 'Fiber optic', 'OnlineSecurity': 'No', 'OnlineBackup': 'No', 'DeviceProtection': 'No', 'TechSupport': 'No', 'StreamingTV': 'No', 'StreamingMovies': 'No', 'Contract': 'Month-to-month', 'PaperlessBilling': 'Yes', 'PaymentMethod': 'Electronic check', 'MonthlyCharges': 75.3, 'TotalCharges': ''}
results = predict_all_models(synthetic_new_customer, artifacts)
print(format_prediction_report(synthetic_new_customer, results))
print()
print('No errors -- the pipeline correctly handles a genuinely new tenure=0 customer.')

In [ ]:
new_customer = {'customerID': 'CUSTOMER-EXAMPLE', 'gender': 'Female', 'SeniorCitizen': 0, 'Partner': 'Yes', 'Dependents': 'No', 'tenure': 5, 'PhoneService': 'Yes', 'MultipleLines': 'No', 'InternetService': 'Fiber optic', 'OnlineSecurity': 'No', 'OnlineBackup': 'No', 'DeviceProtection': 'No', 'TechSupport': 'No', 'StreamingTV': 'Yes', 'StreamingMovies': 'Yes', 'Contract': 'Month-to-month', 'PaperlessBilling': 'Yes', 'PaymentMethod': 'Electronic check', 'MonthlyCharges': 95.5, 'TotalCharges': 477.5}

In [ ]:
results = predict_all_models(new_customer, artifacts)
print(format_prediction_report(new_customer, results))